# 데이터 불러오기와 구조 확인

| 셀 | 코드 | 역할 |
|---|---|---|
| 1 | `read_json`, `shape` | 두 파일을 DataFrame으로 읽고 크기(행, 열)를 확인한다. `max_colwidth`는 긴 claim이 잘리지 않게 하는 화면 설정이다. |
| 2 | `fever.head()` | FEVER 앞 5행으로 열 구성과 실제 claim 모양을 확인한다. |
| 3 | `hover.head()` | HOVER 앞 5행을 확인한다. 같은 hpqa_id에서 단어만 바꾼 변형 claim이 있다. |
| 4 | `dtypes` | 열별 자료형을 확인한다. `evidence`, `supporting_facts`가 object(중첩 리스트)라서 펼쳐야 분석할 수 있다. |

In [1]:
import json
import pandas as pd

fever = pd.read_json("../data/raw/FEVER.jsonl", lines=True)
hover = pd.read_json("../data/raw/HOVER.json")

pd.set_option("display.max_colwidth", 200)  # claim 전문 보기
fever.shape, hover.shape

((145449, 5), (18171, 6))

In [ ]:
fever.head()

,id,verifiable,label,claim,evidence
0,75397,VERIFIABLE,SUPPORTS,Nikolaj Coster-Waldau worked with the Fox Broadcasting Company.,"[[[92206, 104971, Nikolaj_Coster-Waldau, 7], [92206, 104971, Fox_Broadcasting_Company, 0]]]"
1,150448,VERIFIABLE,SUPPORTS,Roman Atwood is a content creator.,"[[[174271, 187498, Roman_Atwood, 1]], [[174271, 187499, Roman_Atwood, 3]]]"
2,214861,VERIFIABLE,SUPPORTS,"History of art includes architecture, dance, sculpture, music, painting, poetry literature, theatre, narrative, film, photography and graphic arts.","[[[255136, 254645, History_of_art, 2]]]"
3,156709,VERIFIABLE,REFUTES,Adrienne Bailon is an accountant.,"[[[180804, 193183, Adrienne_Bailon, 0]]]"
4,83235,NOT VERIFIABLE,NOT ENOUGH INFO,System of a Down briefly disbanded in limbo.,"[[[100277, None, None, None]]]"


In [ ]:
hover.head()

,uid,claim,supporting_facts,label,num_hops,hpqa_id
0,330ca632-e83f-4011-b11b-0d0158145036,Skagen Painter Peder Severin Krøyer favored naturalism along with Theodor Esbern Philipsen and the artist Ossian Elgström studied with in the early 1900s.,"[[Kristian Zahrtmann, 0], [Kristian Zahrtmann, 1], [Peder Severin Krøyer, 1], [Ossian Elgström, 2]]",SUPPORTED,3,5ab7a86d5542995dae37e986
1,ca7db8e1-799a-4498-b31b-b1a430b5874e,Skagen Painter Peder Severin Krøyer favored naturalism along with Theodor Esbern Philipsen and the artist Ossian Elgström studied with in 1907.,"[[Kristian Zahrtmann, 0], [Kristian Zahrtmann, 1], [Peder Severin Krøyer, 1], [Ossian Elgström, 2]]",SUPPORTED,3,5ab7a86d5542995dae37e986
2,2e9cf58a-40e1-47ee-aaa8-3b84ebe94cd0,Skagen Painter Peder Severin Krøyer favored naturalism along with Theodor Esbern Philipsen and the muralist Ossian Elgström studied with in 1907.,"[[Kristian Zahrtmann, 0], [Kristian Zahrtmann, 1], [Peder Severin Krøyer, 1], [Ossian Elgström, 2]]",NOT_SUPPORTED,3,5ab7a86d5542995dae37e986
3,f2f4b69d-d833-4f6d-a630-f156a2559af6,"Skagen Painter, who painted the 1893 painting Roses, favored naturalism. Theodor Esbern Philipsen and the artist that Ossian Elgström studied with in December of 1907 also favored naturalism.","[[Kristian Zahrtmann, 0], [Kristian Zahrtmann, 1], [Peder Severin Krøyer, 1], [Ossian Elgström, 2], [Skagens Museum, 1]]",NOT_SUPPORTED,4,5ab7a86d5542995dae37e986
4,824c8ffa-da36-45ea-9f68-0342e3893a63,Skagen Painter Peder Severin Krøyer favored naturalism along with Norderhov and the artist Ossian Elgström studied with in 1907.,"[[Kristian Zahrtmann, 0], [Kristian Zahrtmann, 1], [Peder Severin Krøyer, 1], [Ossian Elgström, 2]]",NOT_SUPPORTED,3,5ab7a86d5542995dae37e986


In [ ]:
fever.dtypes , hover.dtypes

(id             int64
 verifiable       str
 label            str
 claim            str
 evidence      object
 dtype: object,
 uid                    str
 claim                  str
 supporting_facts    object
 label                  str
 num_hops             int64
 hpqa_id                str
 dtype: object)

# 라벨 분포

| 셀 | 코드 | 역할과 결과 |
|---|---|---|
| 5 | FEVER `value_counts` | 라벨별 개수를 센다. SUPPORTS 55%로 불균형하다. |
| 6 | HOVER `value_counts` | SUPPORTED 61%, NOT_SUPPORTED 39%이다. |
| 7 | `crosstab(num_hops, label)` | 홉 × 라벨 교차표. 홉이 늘수록 NOT_SUPPORTED 비율이 커진다 (2홉 28% → 4홉 59%). |

In [ ]:
fever["label"].value_counts()

label
SUPPORTS           80035
NOT ENOUGH INFO    35639
REFUTES            29775
Name: count, dtype: int64

In [ ]:
hover["label"].value_counts()

label
SUPPORTED        11023
NOT_SUPPORTED     7148
Name: count, dtype: int64

In [ ]:
pd.crosstab(hover["num_hops"], hover["label"], margins=True)

label,NOT_SUPPORTED,SUPPORTED,All
num_hops,,,
2,2556,6496,9052
3,2813,3271,6084
4,1779,1256,3035
All,7148,11023,18171


# 근거 구조 파악과 변환

| 셀 | 코드 | 역할 |
|---|---|---|
| 8 | `fever.loc[0, "evidence"]` | 근거 하나를 직접 열어 3중 리스트 구조를 확인한다. |
| 9 | `fever_ev`, `hover_ev` 생성 | 중첩된 근거를 한 행에 근거 문장 하나인 표로 펼친다 (263,822행 / 55,816행). 문장 위치 분석(11번)에 쓴다. |

In [ ]:
fever.loc[0, "evidence"]

[[[92206, 104971, 'Nikolaj_Coster-Waldau', 7],
  [92206, 104971, 'Fox_Broadcasting_Company', 0]]]

In [ ]:
rows = []
for _, r in fever.iterrows():
    for set_idx, ev_set in enumerate(r["evidence"]):
        for _, _, page, sent in ev_set:
            if page is not None:
                rows.append({"id": r["id"], "label": r["label"],
                             "set_idx": set_idx, "page": page, "sent_id": sent})
fever_ev = pd.DataFrame(rows)

hover_ev = hover[["uid", "label", "num_hops", "supporting_facts"]].explode("supporting_facts")
hover_ev[["page", "sent_id"]] = pd.DataFrame(hover_ev["supporting_facts"].tolist(), index=hover_ev.index)

fever_ev.shape, hover_ev.shape

((263822, 5), (55816, 6))

# EDA 분석

| 셀 | 분석 | 결과 | 시사점 |
|---|---|---|---|
| 10 | claim 길이 | FEVER는 라벨별 평균 약 8단어로 같다. HOVER는 홉이 늘수록 길어지고(2홉 18~23 → 4홉 31~32단어), NOT_SUPPORTED가 더 길다. | HOVER는 길이만으로도 라벨을 일부 추측할 수 있으므로 홉별로 평가한다. |
| 11 | 근거 문장 위치 | 0번 문장 비율이 FEVER 36%, HOVER 54%이다. | 문서 앞부분 위주로 후보 문장을 줄일 수 있다. |
| 12 | 부정어 편향 | 부정어 포함 비율이 REFUTES 28.6%, SUPPORTS 1.5%, NEI 2.8%이다. | claim만 보고 맞히는 지름길이 있으므로 claim-only 기준선을 함께 보고한다. |
| 13 | HOVER 변형 그룹 | hpqa_id 6,109개 중 3,537개에서 라벨이 섞여 있다. 그룹당 claim은 1~23개이다. | 검증 셋은 hpqa_id 단위로 나눠야 누수가 없다. |

In [ ]:
#claim 길이 | FEVER는 라벨별 길이가 거의 같고, HOVER는 홉이 늘수록, 그리고 NOT_SUPPORTED일수록 길어집니다.
fever["n_words"] = fever["claim"].str.split().str.len()
hover["n_words"] = hover["claim"].str.split().str.len()
display(fever.groupby("label")["n_words"].describe())
display(hover.groupby(["num_hops", "label"])["n_words"].mean().unstack())

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
NOT ENOUGH INFO,35639.0,8.062768,3.062972,2.0,6.0,7.0,10.0,47.0
REFUTES,29775.0,8.148346,2.766365,3.0,6.0,8.0,10.0,39.0
SUPPORTS,80035.0,8.101118,3.412112,2.0,6.0,7.0,10.0,65.0


label,NOT_SUPPORTED,SUPPORTED
num_hops,,
2,22.616197,18.027094
3,24.558478,22.683889
4,32.286678,30.594745


In [ ]:
#근거 문장 위치 | 근거가 문서 첫 문장(0번)에 몰려 있습니다.
display(fever_ev["sent_id"].value_counts(normalize=True).sort_index().head(10))
display(hover_ev["sent_id"].value_counts(normalize=True).sort_index().head(10))

sent_id
0    0.362096
1    0.109468
2    0.062277
3    0.041335
4    0.032367
5    0.037374
6    0.037882
7    0.035604
8    0.031764
9    0.027261
Name: proportion, dtype: float64

sent_id
0    0.542909
1    0.245216
2    0.118353
3    0.055540
4    0.021840
5    0.009979
6    0.003870
7    0.001326
8    0.000609
9    0.000179
Name: proportion, dtype: float64

In [ ]:
#부정어 편향 | REFUTES의 약 30%에 부정어가 있지만, SUPPORTS는 2% 정도입니다. claim만 보고 라벨을 맞히는 지름길
neg = r"\b(?:not|never|no|only|none|unable|refused|incapable)\b|n't"
fever["has_neg"] = fever["claim"].str.contains(neg, case=False, regex=True)
display(fever.groupby("label")["has_neg"].mean())

label
NOT ENOUGH INFO    0.027610
REFUTES            0.285609
SUPPORTS           0.015068
Name: has_neg, dtype: float64

In [ ]:
display(fever["label"].value_counts(normalize=True))
display(pd.crosstab(hover["num_hops"], hover["label"], normalize="index"))

label
SUPPORTS           0.550262
NOT ENOUGH INFO    0.245027
REFUTES            0.204711
Name: proportion, dtype: float64

label,NOT_SUPPORTED,SUPPORTED
num_hops,,
2,0.282369,0.717631
3,0.462360,0.537640
4,0.586161,0.413839


In [ ]:
#HOVER 변형 그룹 |대부분의 그룹에서 라벨이 섞여 있습니다. 무작위로 나누면 비슷한 claim이 학습과 평가 양쪽에 들어갑니다.
g = hover.groupby("hpqa_id")
display(g.size().value_counts().sort_index())
mixed = g["label"].nunique().gt(1).sum()
print(f"라벨이 섞인 hpqa_id: {mixed} / {g.ngroups}")

1     2328
2     1486
3      495
4      416
5      387
6      318
7      247
8      160
9       95
10      77
11      41
12      27
13      11
14      10
15       3
16       4
17       1
19       2
23       1
Name: count, dtype: int64

라벨이 섞인 hpqa_id: 3537 / 6109


# FEVER 중복·라벨 충돌 claim
## 중복 claim
- claim 열의 문자열이 완전히 같은 행이 2개 이상이면 중복입니다.
- keep=False는 첫 번째 것까지 포함해서 같은 문장의 모든 행을 남긴다는 뜻입니다. 그래서 "중복 행 15,798개"는 중복 문장이 등장한 행 전체이고, 서로 다른 문장으로는 5,894개입니다.
- id는 모두 다릅니다(중복 id 0개). 같은 문장이 서로 다른 항목으로 여러 번 들어 있다는 뜻입니다

## 라벨 충돌 claim
- 같은 문장끼리 묶어서 라벨 종류를 셉니다.
- 라벨이 2종류 이상이면 충돌로 봅니다. 예를 들어 같은 문장이 한 번은 SUPPORTS, 한 번은 NEI면 충돌입니다.

In [ ]:
# 같은 문장이 2번 이상 나온 claim
dup = fever[fever.duplicated("claim", keep=False)]
print(f"중복 행: {len(dup):,} / 고유 중복 claim: {dup['claim'].nunique():,}")

# 그중 라벨이 서로 다른 claim
labels_per_claim = dup.groupby("claim")["label"].agg(lambda s: tuple(sorted(set(s))))
conflict = labels_per_claim[labels_per_claim.str.len() > 1]
print(f"라벨 충돌 claim: {len(conflict):,}")
display(conflict.value_counts())
display(conflict.head(5))

중복 행: 15,798 / 고유 중복 claim: 5,894
라벨 충돌 claim: 753


label
(NOT ENOUGH INFO, REFUTES)              382
(NOT ENOUGH INFO, SUPPORTS)             285
(REFUTES, SUPPORTS)                      77
(NOT ENOUGH INFO, REFUTES, SUPPORTS)      9
Name: count, dtype: int64

claim
21 Jump Street was released by a production company.    (NOT ENOUGH INFO, SUPPORTS)
21 Jump Street was released in the United States.       (NOT ENOUGH INFO, SUPPORTS)
24 is a historical fiction film.                         (NOT ENOUGH INFO, REFUTES)
24 is a thriller television show.                               (REFUTES, SUPPORTS)
50 First Dates is a horror film.                         (NOT ENOUGH INFO, REFUTES)
Name: label, dtype: object

# HOVER 변형 쌍 분석
같은 hpqa_id, 같은 홉 안에서 라벨이 다른 claim 쌍을 만들고, 단어 단위로 무엇이 바뀌었는지 뽑습니다.

FEVER와 달리 HOVER는 부정어로 라벨을 뒤집는 경우가 드뭅니다. 부정어 지름길이 거의 통하지 않는 데이터입니다.
단어 1~2개 차이로 라벨이 뒤집히는 쌍이 수천 개입니다. 모델이 근거를 정밀하게 대조하는지 시험하는 대조 평가 세트로 쓸 수 있습니다.
분류는 대문자, 숫자 같은 규칙 기반이라 거칩니다. 예를 들어 문장 첫 단어가 바뀌어도 entity로 잡힙니다. 그러니 sample(10) 결과를 직접 몇 개 읽어 보고 분류가 맞는지 확인하세요.

In [ ]:
import difflib, itertools, re

def word_diff(a, b):
    ta, tb = a.split(), b.split()
    ops = difflib.SequenceMatcher(None, ta, tb).get_opcodes()
    changes = [(op, " ".join(ta[i1:i2]), " ".join(tb[j1:j2]))
               for op, i1, i2, j1, j2 in ops if op != "equal"]
    n_changed = sum(max(i2 - i1, j2 - j1) for op, i1, i2, j1, j2 in ops if op != "equal")
    return changes, n_changed

rows = []
for hid, g in hover.groupby("hpqa_id"):
    for (_, a), (_, b) in itertools.combinations(g.iterrows(), 2):
        if a["num_hops"] != b["num_hops"] or a["label"] == b["label"]:
            continue
        changes, n = word_diff(a["claim"], b["claim"])
        rows.append({"hpqa_id": hid, "num_hops": a["num_hops"],
                     "claim_a": a["claim"], "label_a": a["label"],
                     "claim_b": b["claim"], "label_b": b["label"],
                     "n_changed": n, "changes": changes})
pairs = pd.DataFrame(rows)
print(f"라벨이 다른 쌍: {len(pairs):,}")
display(pairs["n_changed"].describe())

라벨이 다른 쌍: 10,060


count    10060.000000
mean         4.439563
std          6.896752
min          0.000000
25%          1.000000
50%          2.000000
75%          4.000000
max         52.000000
Name: n_changed, dtype: float64

In [ ]:
# 이어서 바꾼 단어가 3개 이하인 쌍만 골라 변형 유형을 규칙으로 분류합니다.

def change_type(changes):
    text = " ".join(old + " " + new for _, old, new in changes)
    if re.search(r"\d", text):
        return "number_date"
    if re.search(r"\b(?:not|never|no|neither|nor)\b|n't", text, re.I):
        return "negation"
    if any(re.search(r"\b[A-Z]", old + new) for _, old, new in changes):
        return "entity"
    return "other_word"

small = pairs[pairs["n_changed"].between(1, 3)].copy()
small["change_type"] = small["changes"].apply(change_type)
print(f"3단어 이하로 바뀐 쌍: {len(small):,}")
display(small["change_type"].value_counts(normalize=True))
display(small.sample(10, random_state=0)[["changes", "label_a", "label_b"]])

3단어 이하로 바뀐 쌍: 7,228


change_type
entity         0.483536
other_word     0.366630
number_date    0.109436
negation       0.040398
Name: proportion, dtype: float64

,changes,label_a,label_b
1184,"[(insert, , 1950s)]",SUPPORTED,NOT_SUPPORTED
7720,"[(replace, competed, qualified)]",SUPPORTED,NOT_SUPPORTED
5644,"[(delete, comedy, )]",NOT_SUPPORTED,SUPPORTED
4313,"[(delete, coastal, )]",NOT_SUPPORTED,SUPPORTED
8351,"[(replace, American, English)]",NOT_SUPPORTED,SUPPORTED
5011,"[(replace, Veyyil, Robert De Niro)]",SUPPORTED,NOT_SUPPORTED
7758,"[(delete, not, ), (insert, , not)]",NOT_SUPPORTED,SUPPORTED
6317,"[(replace, partnered, represented), (replace, sixth, fifth)]",SUPPORTED,NOT_SUPPORTED
9760,"[(delete, Japanese horror, )]",NOT_SUPPORTED,SUPPORTED
3013,"[(insert, , (1950 film))]",SUPPORTED,NOT_SUPPORTED


HOVER에도 문장이 완전히 같은데 라벨이 다른 claim이 8쌍 있습니다
개수가 적으니 학습에서 빼는 게 안전합니다.

In [ ]:
hd = hover[hover.duplicated("claim", keep=False)]
c = hd.groupby("claim")["label"].nunique()
display(hd[hd["claim"].isin(c[c > 1].index)].sort_values("claim")[["claim", "label", "num_hops"]])

,claim,label,num_hops
4060,"A star of Cassadaga (film) was part of the cast of the 2015 American drama film, written and directed by Tony Aloupis.",NOT_SUPPORTED,3
4061,"A star of Cassadaga (film) was part of the cast of the 2015 American drama film, written and directed by Tony Aloupis.",SUPPORTED,3
9394,ICI House is now named after the company that provides explosives.,SUPPORTED,2
9395,ICI House is now named after the company that provides explosives.,NOT_SUPPORTED,2
9821,The Los Angeles Dance Theater production company was founded by George Balanchine to create a live stage version of the 1942 film Casablanca.,NOT_SUPPORTED,2
9822,The Los Angeles Dance Theater production company was founded by George Balanchine to create a live stage version of the 1942 film Casablanca.,SUPPORTED,2
7945,"The actor who had his voice featured in the 2012 film The Polar Bears, stars as Oliver in a film directed by Luca Guadagnino.",NOT_SUPPORTED,3
7947,"The actor who had his voice featured in the 2012 film The Polar Bears, stars as Oliver in a film directed by Luca Guadagnino.",SUPPORTED,3
17977,The home of Sir John Cutts who died in 1615 is a house at a parish 18.7 miles from central London.,NOT_SUPPORTED,3
17978,The home of Sir John Cutts who died in 1615 is a house at a parish 18.7 miles from central London.,SUPPORTED,3


### FEVER
* 145,449개 claim, 5개 열 (`id`, `verifiable`, `label`, `claim`, `evidence`)
* label: SUPPORTS 55.0% / NOT ENOUGH INFO 24.5% / REFUTES 20.5% (불균형)
* `verifiable`은 label에서 바로 나오는 값이다 (NEI → NOT VERIFIABLE, 나머지 → VERIFIABLE)
* 대부분 단일 문서로 판정할 수 있다 (근거 문서 2개 이상 필요한 claim은 약 10%)
* REFUTES에 부정어 편향이 있다 (부정어 포함 비율: REFUTES 28.6%, SUPPORTS 1.5%)
* 같은 문장이 여러 번 나오는 claim이 5,894개이고, 그중 753개는 라벨이 다르다 (대부분 NEI가 섞인 충돌)

### HOVER
* 18,171개 claim, 6개 열 (`uid`, `claim`, `supporting_facts`, `label`, `num_hops`, `hpqa_id`)
* label: SUPPORTED 60.7% / NOT_SUPPORTED 39.3%
* 멀티홉 (2홉 49.8% / 3홉 33.5% / 4홉 16.7%), 홉 수 = 필요한 근거 문서 수
* 홉이 늘수록 NOT_SUPPORTED 비율이 커진다 (2홉 28% → 4홉 59%)
* 홉이 늘수록 claim이 길어지고, 같은 홉에서도 NOT_SUPPORTED가 더 길다
* 같은 hpqa_id(원본 HotpotQA 문항)에서 단어만 바꾼 변형 claim이 많다 (3,781개 그룹 중 3,537개에서 라벨이 섞임)

### 공통
* 근거는 위치(문서명, 문장번호)만 있고 본문이 없다 → 위키 코퍼스가 필요하다
* 근거가 문서 앞부분에 몰려 있다 (0번 문장 비율: FEVER 36%, HOVER 54%)
* 라벨 체계가 다르다: HOVER의 NOT_SUPPORTED = FEVER의 REFUTES + NOT ENOUGH INFO
* claim 문장만으로 라벨을 일부 맞힐 수 있는 지름길이 있다 (FEVER는 부정어, HOVER는 길이)